In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/content/aggregrated_data.CSV")

In [ ]:
print(df.shape)
print(df.info())
print((df == 0).sum())

(2160, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2160 entries, 0 to 2159
Data columns (total 20 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Time(ms)     2160 non-null   int64  
 1   Temp(C)      2148 non-null   float64
 2   Humidity(%)  2148 non-null   float64
 3   RainAnalog   2160 non-null   int64  
 4   RainDigital  2160 non-null   int64  
 5   SNR1         2160 non-null   int64  
 6   SNR2         2160 non-null   int64  
 7   SNR3         2160 non-null   int64  
 8   SNR4         2160 non-null   int64  
 9   SNR5         2160 non-null   int64  
 10  SNR6         2160 non-null   int64  
 11  SNR7         2160 non-null   int64  
 12  SNR8         2160 non-null   int64  
 13  SNR9         2160 non-null   int64  
 14  SNR10        2160 non-null   int64  
 15  SNR11        2160 non-null   int64  
 16  SNR12        2160 non-null   int64  
 17  SNR13        2160 non-null   int64  
 18  SNR14        2160 non-null   int64  


In [ ]:
df["RainDigital"] = 0

In [ ]:
snr_cols = [col for col in df.columns if "SNR" in col]

for col in snr_cols:
    df.loc[df[col] > 100, col] = np.nan
    df.loc[df[col] < 10, col] = np.nan

In [ ]:
for col in snr_cols:
    df.loc[df[col] == 0, col] = np.nan

In [ ]:
df.loc[df["Temp(C)"] == 0, "Temp(C)"] = np.nan
df.loc[df["Humidity(%)"] == 0, "Humidity(%)"] = np.nan

In [ ]:
df = df.sort_values("Time(ms)")

In [ ]:
df[snr_cols] = df[snr_cols].interpolate(method="linear")
df[["Temp(C)", "Humidity(%)"]] = df[["Temp(C)", "Humidity(%)"]].interpolate()

In [ ]:
df.fillna(method="bfill", inplace=True)
df.fillna(method="ffill", inplace=True)

/tmp/ipython-input-2360960147.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="bfill", inplace=True)
/tmp/ipython-input-2360960147.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="ffill", inplace=True)


In [ ]:
for col in snr_cols:
    df[col] = df[col].rolling(window=3, min_periods=1).mean()

In [ ]:
for col in snr_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

In [ ]:
df[snr_cols] = df[snr_cols].round().astype("Int64")

In [ ]:
df["Mean_SNR"] = df[snr_cols].mean(axis=1)

In [ ]:
df["SNR_Var"] = df[snr_cols].var(axis=1)

In [ ]:
df["Sat_Count"] = df[snr_cols].notna().sum(axis=1)

In [ ]:
df["SNR_Diff"] = df["Mean_SNR"].diff().fillna(0)

In [ ]:
df.to_csv("Sunny_Clean.csv", index=False)